# VocEd Lab 06 — U-Net: Multi-Scale Segmentation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emilsar/VocEd/blob/main/06_unet.ipynb)

The minimal CNN in Lab 05 processes every image at a fixed 256×256 resolution.  Its filters
can only "see" a small neighbourhood of pixels — they never zoom out to understand the
global structure of a cell.  This leads to mistakes at large structures: the network can
correctly classify individual pixels but misses the overall shape.

The **U-Net** (Ronneberger et al., 2015) fixes this with an **encoder-decoder** design:
- The **encoder** progressively downsamples the image (128→64→32...), building a representation
  that captures global context.
- The **decoder** progressively upsamples back to the original resolution, recovering fine
  spatial detail.
- **Skip connections** copy feature maps from each encoder level to the corresponding
  decoder level, so local detail is not lost during downsampling.

By the end of this lab you will be able to:
- Explain the encoder-decoder architecture and the role of skip connections.
- Build a small U-Net in PyTorch with 2 encoder levels, a bottleneck, and 2 decoder levels.
- Train it with a combined **cross-entropy + Dice** loss — and explain why a U-Net benefits from the Dice term where Lab 05's per-pixel CNN did not.
- Report the final entry in the cumulative Dice table.
- Save model weights for use in Lab 07.

## 0. Setup

In [ ]:
# Clone the repo
!git clone https://github.com/emilsar/VocEd.git
%cd VocEd

## 1. Load Data & Recreate the Train/Test Split

In [ ]:
import glob
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from sklearn.model_selection import train_test_split

N = len(glob.glob('imagedata/X/*.npy'))
X = np.stack([np.load(f'imagedata/X/{i}.npy') for i in range(N)])
y = np.stack([np.load(f'imagedata/y/{i}.npy') for i in range(N)])

# Drop images with no nucleus pixels — identical to Lab 02
has_nucleus = (y == 2).sum(axis=(1, 2)) > 0
X, y = X[has_nucleus], y[has_nucleus]
N = len(X)

# Stratified split by N/C ratio — identical to Lab 02
nuc_px   = (y == 2).sum(axis=(1, 2))
cyt_px   = (y == 1).sum(axis=(1, 2))
nc_ratio = nuc_px / np.maximum(cyt_px, 1)   # np.maximum avoids div-by-zero
quartile = np.digitize(nc_ratio, np.percentile(nc_ratio, [25, 50, 75]))

train_idx, test_idx = train_test_split(
    np.arange(N), test_size=0.2, stratify=quartile, random_state=42
)

mask_cmap = ListedColormap(['black', 'steelblue', 'crimson'])
legend_patches = [
    mpatches.Patch(color='black',     label='0 — background'),
    mpatches.Patch(color='steelblue', label='1 — cytoplasm'),
    mpatches.Patch(color='crimson',   label='2 — nucleus'),
]
print(f'{N} images retained  ({(~has_nucleus).sum()} with no nucleus removed)')
print(f'Train: {len(train_idx)}   Test: {len(test_idx)}')

## 2. U-Net Architecture

```
Input (N, 3, 256, 256)
│
├── Encoder level 1 → (N, 32, 256, 256) ──────────────────── skip 1 ──┐
│       MaxPool 2×2 → (N, 32, 128, 128)                                │
├── Encoder level 2 → (N, 64, 128, 128) ──────────── skip 2 ──┐       │
│       MaxPool 2×2 → (N, 64, 64, 64)                          │       │
├── Bottleneck      → (N, 128, 64, 64)                         │       │
│       Upsample 2× → (N, 128, 128, 128)                       │       │
├── Decoder level 2: concat skip2 → (N, 192, 128, 128)         ┘       │
│                    conv → (N, 64, 128, 128)                           │
│       Upsample 2× → (N, 64, 256, 256)                                │
├── Decoder level 1: concat skip1 → (N, 96, 256, 256)                  ┘
│                    conv → (N, 32, 256, 256)
└── Output head 1×1 conv → (N, 3, 256, 256)  ← class logits
```

Each "conv block" contains two `Conv2d → BatchNorm → ReLU` steps.  Batch normalisation
stabilises training by normalising each feature map within a batch.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)


# ── Re-usable conv block ──────────────────────────────────────────────────────
# Two Conv2d → BatchNorm2d → ReLU sequences.
# BatchNorm2d normalises the activations across the batch — helps training converge faster.
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


# ── Small U-Net ───────────────────────────────────────────────────────────────
class SmallUNet(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()

        # Encoder: each level halves the spatial size
        self.enc1 = ConvBlock(3,   32)    # 256 → still 256 (MaxPool below does the halving)
        self.enc2 = ConvBlock(32,  64)    # 128 → still 128

        # MaxPool2d(2) halves both height and width
        self.pool = nn.MaxPool2d(2)

        # Bottleneck: deepest part of the U
        self.bottleneck = ConvBlock(64, 128)   # operates at 64×64

        # Decoder: each level doubles the spatial size via bilinear upsampling
        # After upsampling we concatenate the skip connection from the encoder,
        # which doubles the number of channels — that's why in_ch = upsample_ch + skip_ch.
        self.dec2 = ConvBlock(128 + 64,  64)   # 128 from bottleneck + 64 from enc2 skip
        self.dec1 = ConvBlock(64  + 32,  32)   # 64  from dec2       + 32 from enc1 skip

        # Output head: 1×1 conv maps 32 channels → num_classes logits
        self.out_conv = nn.Conv2d(32, num_classes, kernel_size=1)

    def forward(self, x):
        # ── Encoder ──────────────────────────────────────────────────────────
        s1 = self.enc1(x)           # (N,  32, 256, 256) — save for skip connection
        s2 = self.enc2(self.pool(s1))  # (N,  64, 128, 128) — save for skip connection

        # ── Bottleneck ────────────────────────────────────────────────────────
        b  = self.bottleneck(self.pool(s2))  # (N, 128, 64, 64)

        # ── Decoder ──────────────────────────────────────────────────────────
        # F.interpolate doubles spatial size using bilinear interpolation
        u2 = F.interpolate(b,  scale_factor=2, mode='bilinear', align_corners=False)
        # torch.cat concatenates along the channel dimension (dim=1)
        u2 = self.dec2(torch.cat([u2, s2], dim=1))  # (N, 64, 128, 128)

        u1 = F.interpolate(u2, scale_factor=2, mode='bilinear', align_corners=False)
        u1 = self.dec1(torch.cat([u1, s1], dim=1))  # (N, 32, 256, 256)

        return self.out_conv(u1)   # (N, 3, 256, 256)


model = SmallUNet(num_classes=3).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {total_params:,}')

## 3. Loss Function: Cross-Entropy + Dice

Lab 05 trained its tiny CNN with **cross-entropy** alone and that was enough.  Why does
the U-Net need something extra?

Cross-entropy is computed **per pixel**: each pixel is judged on its own, independently
of its neighbours.  Lab 05's network was a stack of three convolutions with no pooling —
essentially a learned colour rule per pixel — so there was no way for it to merge two
visually similar classes.  The U-Net, by contrast, has the spatial reach (via pooling +
skip connections) to *decide regions*, not just pixels.  In urothelial cytology the
background and cytoplasm are both light-toned, so when the U-Net is uncertain it tends to
collapse background into the larger cytoplasm class — a failure mode that costs almost no
cross-entropy but tanks the Dice score.

The fix is to add a **Dice loss** term, which scores predictions on **mask overlap**
rather than per-pixel correctness:

- **Cross-entropy loss** — standard classification loss; penalises wrong class probabilities.
- **Dice loss** — directly penalises overlap error between predicted and target masks;
  correlates directly with the metric we report.

Adding them with equal weight gives the U-Net both per-pixel supervision and a region-level
signal that prevents bg/cyt collapse.

In [ ]:
def dice_loss(logits, targets, num_classes=3, smooth=1.0):
    """
    Soft Dice loss averaged over all classes.

    logits  : (N, C, H, W) raw output from the network (before softmax)
    targets : (N, H, W) integer class labels
    smooth  : small constant to avoid division by zero
    """
    # softmax converts logits into per-pixel class probabilities
    probs = torch.softmax(logits, dim=1)   # (N, C, H, W)

    # Convert integer labels to one-hot: (N, H, W) → (N, C, H, W)
    # one_hot returns (N, H, W, C), so we permute to (N, C, H, W)
    targets_onehot = F.one_hot(targets, num_classes).permute(0, 3, 1, 2).float()

    # Compute Dice per class, then average
    loss = 0.0
    for cls in range(num_classes):
        p = probs[:, cls]              # (N, H, W) probability for this class
        t = targets_onehot[:, cls]     # (N, H, W) ground-truth binary mask
        intersection = (p * t).sum()
        denom        = p.sum() + t.sum()
        # 1 - Dice because we want to *minimise* the loss
        loss += 1.0 - (2.0 * intersection + smooth) / (denom + smooth)
    return loss / num_classes


ce_loss = nn.CrossEntropyLoss()

def combined_loss(logits, targets):
    # Add cross-entropy and Dice loss with equal weight
    return ce_loss(logits, targets) + dice_loss(logits, targets)


print('Loss functions defined.')

## 4. Dataset & Training

In [ ]:
class SegDataset(Dataset):
    def __init__(self, X, y, indices):
        self.X = X
        self.y = y
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        img  = torch.from_numpy(self.X[idx])
        mask = torch.from_numpy(self.y[idx].astype(np.int64))
        return img, mask


train_loader = DataLoader(SegDataset(X, y, train_idx), batch_size=8, shuffle=True)
test_loader  = DataLoader(SegDataset(X, y, test_idx),  batch_size=8, shuffle=False)

optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

# ── Training loop ─────────────────────────────────────────────────────────────
NUM_EPOCHS = 10
train_losses = []

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0.0

    for imgs, masks in train_loader:
        imgs  = imgs.to(device)
        masks = masks.to(device)

        logits = model(imgs)
        loss   = combined_loss(logits, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    print(f'Epoch {epoch+1:2d}/{NUM_EPOCHS}  —  avg loss: {avg_loss:.4f}')

# ── Save the trained weights ──────────────────────────────────────────────────
# torch.save saves the model's state dictionary (all learnable parameters)
torch.save(model.state_dict(), 'unet_weights.pt')
print('\nWeights saved to unet_weights.pt')

# ── Loss curve ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, NUM_EPOCHS + 1), train_losses, marker='o', color='steelblue')
ax.set_xlabel('Epoch')
ax.set_ylabel('Combined loss (CE + Dice)')
ax.set_title('Training Loss Curve — Small U-Net')
plt.tight_layout()
plt.show()

## 5. Evaluation & Cumulative Dice Table

In [ ]:
def dice_score_np(pred, target, cls):
    pred_mask   = (pred   == cls)
    target_mask = (target == cls)
    intersection = (pred_mask & target_mask).sum()
    denom = pred_mask.sum() + target_mask.sum()
    return 1.0 if denom == 0 else 2 * intersection / denom


model.eval()
unet_scores = []

with torch.no_grad():
    for imgs, masks in test_loader:
        imgs   = imgs.to(device)
        logits = model(imgs)
        preds  = logits.argmax(dim=1).cpu().numpy()
        masks  = masks.numpy()
        for pred, mask in zip(preds, masks):
            d = (dice_score_np(pred, mask, 1) + dice_score_np(pred, mask, 2)) / 2
            unet_scores.append(d)

mean_unet = np.mean(unet_scores)
print(f'U-Net test Dice: {mean_unet:.4f}  ±  {np.std(unet_scores):.4f}')

# ── Cumulative table ──────────────────────────────────────────────────────────
def segment_plain(img, t_nucleus=0.45, t_background=0.85):
    gray = 0.299 * img[0] + 0.587 * img[1] + 0.114 * img[2]
    pred = np.zeros(gray.shape, dtype=np.int64)
    pred[gray < t_nucleus]                                = 2
    pred[(gray >= t_nucleus) & (gray < t_background)]    = 1
    return pred

def segment_lab02(img):
    return segment_plain(img, t_nucleus=0.3904, t_background=0.9900)

d_lab01 = np.mean([
    (dice_score_np(segment_plain(X[i]), y[i], 1) +
     dice_score_np(segment_plain(X[i]), y[i], 2)) / 2
    for i in test_idx
])
d_lab02 = np.mean([
    (dice_score_np(segment_lab02(X[i]), y[i], 1) +
     dice_score_np(segment_lab02(X[i]), y[i], 2)) / 2
    for i in test_idx
])

print('\nCumulative Dice comparison (test set):')
print('=' * 55)
print(f'{"Lab 01 — hand-picked thresholds":<38}  {d_lab01:.4f}')
print(f'{"Lab 02 — Bayesian optimisation":<38}  {d_lab02:.4f}')
print(f'{"Lab 06 — Small U-Net":<38}  {mean_unet:.4f}')
print('=' * 55)

In [ ]:
# ── Visual comparison: Lab 05 failure cases ───────────────────────────────────
# Pick test images with the lowest U-Net Dice to see where it still struggles
worst_indices = np.argsort(unet_scores)[:4]   # 4 worst-performing test images

fig, axes = plt.subplots(4, 3, figsize=(12, 16))
for row, rel_idx in enumerate(worst_indices):
    abs_idx = test_idx[rel_idx]

    img_t = torch.from_numpy(X[abs_idx]).unsqueeze(0).to(device)
    with torch.no_grad():
        pred = model(img_t).argmax(dim=1).squeeze().cpu().numpy()

    axes[row, 0].imshow(X[abs_idx].transpose(1, 2, 0))
    axes[row, 0].set_title(f'Image {abs_idx} — RGB');  axes[row, 0].axis('off')

    axes[row, 1].imshow(y[abs_idx], cmap=mask_cmap, vmin=0, vmax=2, interpolation='nearest')
    axes[row, 1].set_title('Ground truth');  axes[row, 1].axis('off')

    d = unet_scores[rel_idx]
    axes[row, 2].imshow(pred, cmap=mask_cmap, vmin=0, vmax=2, interpolation='nearest')
    axes[row, 2].set_title(f'U-Net  Dice={d:.3f}');  axes[row, 2].axis('off')

fig.legend(handles=legend_patches, loc='lower center', ncol=3, fontsize=10,
           bbox_to_anchor=(0.5, 0.0))
plt.suptitle('Worst-performing U-Net test images', fontsize=13, y=1.01)
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()

## 6. N/C Ratio Scatter — Predicted vs True

Dice measures mask overlap.  Here we check whether the U-Net's Dice improvement over
the threshold baseline translates into a more accurate **N/C ratio**.  Points near the
y = x line mean the predicted ratio closely matches the ground truth.

In [ ]:
def r2_identity(yp, gt):
    """R² vs y = x: 1 means perfect agreement with the identity line."""
    ss_res = np.sum((yp - gt) ** 2)
    ss_tot = np.sum((gt - gt.mean()) ** 2)
    return 1 - ss_res / ss_tot

def nc_ratio(mask):
    """N/C ratio: nucleus pixel count / cytoplasm pixel count. Returns nan if no cytoplasm."""
    nuc = (mask == 2).sum()
    cyt = (mask == 1).sum()
    return nuc / cyt if cyt > 0 else np.nan


# ── True N/C ratios from ground-truth masks ───────────────────────────────────
nc_true = np.array([nc_ratio(y[i]) for i in test_idx])

# ── Lab 01: hand-picked threshold predictions ─────────────────────────────────
nc_lab01 = np.array([nc_ratio(segment_plain(X[i])) for i in test_idx])

# ── Lab 02: Bayesian-optimised threshold predictions ─────────────────────────
nc_lab02 = np.array([nc_ratio(segment_lab02(X[i])) for i in test_idx])

# ── U-Net: run inference one image at a time ──────────────────────────────────
model.eval()
nc_unet = []
with torch.no_grad():
    for i in test_idx:
        img_t = torch.from_numpy(X[i]).unsqueeze(0).to(device)
        pred  = model(img_t).argmax(dim=1).squeeze().cpu().numpy()
        nc_unet.append(nc_ratio(pred))
nc_unet = np.array(nc_unet)

# ── Scatter plots ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, nc_pred, label, color in zip(
        axes,
        [nc_lab01, nc_lab02, nc_unet],
        ['Lab 01 — hand-picked', 'Lab 02 — Bayesian opt', 'Lab 06 — Small U-Net'],
        ['steelblue', 'seagreen', 'crimson']):

    valid = np.isfinite(nc_true) & np.isfinite(nc_pred)
    x, yp = nc_true[valid], nc_pred[valid]

    r2 = r2_identity(yp, x)

    ax.scatter(x, yp, alpha=0.6, color=color, edgecolors='none', s=40)
    lim = max(x.max(), yp.max()) * 1.1
    ax.plot([0, lim], [0, lim], 'k--', linewidth=1.5, label='y = x (perfect)')
    ax.set_xlim(0, lim); ax.set_ylim(0, lim)
    ax.set_aspect('equal')
    ax.set_xlabel('True N/C ratio')
    ax.set_ylabel('Predicted N/C ratio')
    ax.set_title(f'{label}\nR² = {r2:.3f}')
    ax.legend(fontsize=9)

plt.suptitle('N/C Ratio: Predicted vs True (test set)', fontsize=13)
plt.tight_layout()
plt.show()

## Wrap-up

**Key takeaways:**
- The encoder-decoder structure with skip connections allows the U-Net to use both global
  context (from deep, small feature maps) and local detail (from shallow, full-resolution
  feature maps) simultaneously.
- Cross-entropy alone is not enough for this network: its spatial reach lets it merge
  visually similar classes (background and cytoplasm), so we add a **Dice** term to keep a
  region-level signal in the loss.
- In Lab 07 we use the trained U-Net to answer the original clinical question: can we
  predict the N/C ratio accurately enough to be useful?

---

## Group Exercise — Skip Connections

Skip connections are the defining feature of U-Net.  What happens without them?

**Person A** — Remove the skip connections.  In `SmallUNet.forward()`, replace
  `torch.cat([u2, s2], dim=1)` with just `u2`, and update `ConvBlock` input sizes
  accordingly.  Train for 10 epochs and record test Dice.

**Person B** — Use the full U-Net (this lab's default).  Record test Dice.

**Person C** — Add a third encoder/decoder level (down to 32×32).  Update channel sizes
  consistently.  Train for 10 epochs and record test Dice and training time.

Share results and discuss: how much do skip connections contribute?  Does going deeper help?